In [12]:
import os
import math
import collections


path = r'C:\Users\NITRO\Desktop\Collection'
folders = ['Gaza', 'Sport', 'Economy'] 
documents = {}

for folder in folders:
    folder_path = os.path.join(path, folder)
    for filename in os.listdir(folder_path):
                file_path = os.path.join(folder_path, filename)
                with open(file_path, 'r', encoding='utf-8') as f:
                        documents[filename] = f.read()
    




#1- Creating a bag of words vector that includes all distinct terms in all documents
all_text = ""
for content in documents.values():
    all_text += content.lower() + " "

words_list = all_text.split()

bag_of_words = sorted(list(set(words_list)))






#2-	Converting all documents into frequency vectors that store the frequency the terms in the bag of words vector in each document.
all_doc_freq_vectors = {}

for doc_name, content in documents.items():
    doc_words = content.lower().split()
    word_counts= {}
    for word in doc_words:
        if word in word_counts:
            word_counts[word] += 1 
        else:
            word_counts[word] = 1  
    
    vector = []
    for word in bag_of_words:
        if word in word_counts:
            vector.append(word_counts[word])
        else:
            vector.append(0)
            
    all_doc_freq_vectors[doc_name] = vector





#3- Construct tf-idf vector for each document.
#IDF
N = len(documents)
df = {word: 0 for word in bag_of_words}

for content in documents.values():
    words = set(content.lower().split())
    for word in words:
        if word in df:
            df[word] += 1

idf = {word: math.log10(N / df[word]) if df[word] > 0 else 0 for word in bag_of_words}

#tf-idf vector for each document.
all_doc_tfidf_vectors = {}

for doc_name, freq_vector in all_doc_freq_vectors.items():
    tfidf_vector = []
    
    for i in range(len(bag_of_words)):
        word = bag_of_words[i]
        tf = freq_vector[i]
        
        tf_weight = math.log10(1 + tf) 
        word_idf = idf[word] 
        weight = tf_weight * word_idf
        tfidf_vector.append(weight)
    
    all_doc_tfidf_vectors[doc_name] = tfidf_vector





#3-	Converting the query into frequency vector, the same like documents vectors.
query = input("\nEnter your search query: ")
query_words = query.lower().split()

query_counts_dict = {}
for word in query_words:
    if word in query_counts_dict:
        query_counts_dict[word] += 1
    else:
        query_counts_dict[word] = 1

query_freq_vector = []
for word in bag_of_words:
    if word in query_counts_dict:
        query_freq_vector.append(query_counts_dict[word])
    else:
        query_freq_vector.append(0)

#query tf-idf vector
query_tfidf_vector = []
for i in range(len(bag_of_words)):
    word = bag_of_words[i]
    tf= query_freq_vector[i]
    
    tf_weight = math.log10(1 + tf)
    query_tfidf_vector.append(tf_weight * idf[word])





#4-	Measure the similarity (Cos, Jaccard) between the query and each document based on tf-idf weighting.
def get_cosine_sim(v1, v2):
    dot_product = sum(a * b for a, b in zip(v1, v2))
    len1 = math.sqrt(sum(a**2 for a in v1))
    len2 = math.sqrt(sum(b**2 for b in v2))
    if len1 * len2 == 0: return 0
    return dot_product / (len1 * len2)

def get_jaccard_set_sim(q_text, d_text):
    set_q = set(q_text.lower().split())
    set_d = set(d_text.lower().split())
    intersection = set_q.intersection(set_d)
    union = set_q.union(set_d)
    if not union: return 0
    return len(intersection) / len(union)

cos_results = []
jaccard_results = []

for doc_name, doc_tfidf in all_doc_tfidf_vectors.items():
    cos_score = get_cosine_sim(query_tfidf_vector, doc_tfidf)
    cos_results.append((doc_name, cos_score))
    
    jac_score = get_jaccard_set_sim(query, documents[doc_name])
    jaccard_results.append((doc_name, jac_score))





#5-	Rank the related documents according to their similarity value with the query.
cos_results.sort(key=lambda x: x[1], reverse=True)
jaccard_results.sort(key=lambda x: x[1], reverse=True)





print("\nThe similarity between query and documents based on Cos similarity is:")
print("Doc id\t\t\tSimilarity")
for doc_id, score in cos_results[:10]:
    print(f"{doc_id}\t\t{score:.2f}")

print("\nThe similarity between query and documents based on Jaccard similarity is:")
print("Doc id\t\t\tSimilarity")
for doc_id, score in jaccard_results[:10]:
    print(f"{doc_id}\t\t{score:.2f}")



Enter your search query:  sport



The similarity between query and documents based on Cos similarity is:
Doc id			Similarity
39 Sport.txt		0.11
28 Sport.txt		0.10
46 Sport.txt		0.10
33 Sport.txt		0.09
26 Sport.txt		0.09
50 Sport.txt		0.08
32 Sport.txt		0.08
29 Sport.txt		0.07
49 Sport.txt		0.05
41 Sport.txt		0.04

The similarity between query and documents based on Jaccard similarity is:
Doc id			Similarity
31 Sport.txt		0.01
33 Sport.txt		0.01
28 Sport.txt		0.01
32 Sport.txt		0.01
38 Sport.txt		0.01
27 Sport.txt		0.01
29 Sport.txt		0.01
34 Sport.txt		0.00
39 Sport.txt		0.00
26 Sport.txt		0.00
